In [1]:
#Unir archivos txt en un solo dataframe y exportar los resultados
import pandas as pd
import glob
import os

ruta_archivos=r'D:\CORREDORES_V_DATASET\IMAGES\ONLY_ROADS_SAMpavement_terrz19_Z21_BB22\seg_col\Troncal1'

# Buscar todos los archivos *_colores.txt
archivos_txt = glob.glob(os.path.join(ruta_archivos, "*_colores.txt"))

# Cargar todos los archivos en un solo DataFrame
dfs = []
for archivo in archivos_txt:
    try:
        df = pd.read_csv(archivo, sep="\t")
        dfs.append(df)
    except Exception as e:
        print(f"❌ Error cargando {archivo}: {e}")

# Combinar todos
df_completo = pd.concat(dfs, ignore_index=True)

# Agrupar por imagen y descripción
df_agrupado = df_completo.groupby(
    ["imagen", "descripcion"], as_index=False
).agg({
    "pixeles": "sum",
    "porcentaje_total_imagen": "sum",
    "porcentaje_filtrada": "sum"
})

out_excel=ruta_archivos+'.xlsx'
df_agrupado.to_excel(out_excel)

# Agrupar por imagen
df_agrupado2 = df_agrupado.groupby(
    ["imagen"], as_index=False
).agg({
    "pixeles": "sum",
    "porcentaje_total_imagen": "sum",
    "porcentaje_filtrada": "sum"
})

# Guardar el resultado agrupado
out_csv=ruta_archivos+'_colores_agrupados.csv'
df_agrupado2.to_csv(out_csv, index=False, sep=",")

df_agrupado2

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_11948\1516940096.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_completo = pd.concat(dfs, ignore_index=True)


,imagen,pixeles,porcentaje_total_imagen,porcentaje_filtrada
0,Troncal1_10000.tif,488,0.0318,99.9994
1,Troncal1_10001.tif,76,0.0055,100.0006
2,Troncal1_10002.tif,356,0.0226,100.0004
3,Troncal1_10005.tif,201,0.0144,99.9979
4,Troncal1_10006.tif,1345,0.0883,99.9920
...,...,...,...,...
6992,Troncal1_9978.tif,16756,1.0438,100.0014
6993,Troncal1_9981.tif,9671,0.7816,99.9707
6994,Troncal1_9992.tif,4934,0.3164,99.9995
6995,Troncal1_9997.tif,34,0.0031,100.0005


In [78]:
#unir segmentacion anomalias de todas las troncales

import pandas as pd
import os, glob
from pathlib import Path

# Ruta de la carpeta donde están los archivos Excel
carpeta = r"D:\CORREDORES_V_DATASET\IMAGES\ONLY_ROADS_SAMpavement_terrz19_Z21_BB22\seg_col"  # Reemplaza con la ruta real
archivos_excel = glob.glob(os.path.join(carpeta,'*.xlsx'))[:6]

# Lista para guardar los DataFrames temporales
lista_dataframes = []

for archivo in archivos_excel:
    try:
        df = pd.read_excel(archivo)
        df['archivo_origen'] = os.path.basename(archivo)  # Opcional: para saber de qué archivo viene cada fila
        lista_dataframes.append(df)
    except Exception as e:
        print(f"Error leyendo {os.path.basename(archivo)}: {e}")

# Unimos todos los DataFrames en uno solo
df_unido = pd.concat(lista_dataframes, ignore_index=True)
df_unido['id'] = df_unido['imagen'].str.split('.', n=1).str[0]


# Mostrar el resultado
print(f"Total de archivos procesados: {len(archivos_excel)}")
print(f"Total de filas unidas: {df_unido.shape[0]}")

excel_full=os.path.join(carpeta,'Troncales_full_segcol.xlsx')
df_unido.to_excel(excel_full)
df_unido


Total de archivos procesados: 6
Total de filas unidas: 131218


,Unnamed: 0,imagen,descripcion,pixeles,porcentaje_total_imagen,porcentaje_filtrada,archivo_origen,id
0,0,Troncal1_10000.tif,Azul marino,62,0.0043,12.7048,Troncal1.xlsx,Troncal1_10000
1,1,Troncal1_10000.tif,Azul oscuro,316,0.0200,64.7539,Troncal1.xlsx,Troncal1_10000
2,2,Troncal1_10000.tif,Azul profundo,43,0.0030,8.8113,Troncal1.xlsx,Troncal1_10000
3,3,Troncal1_10000.tif,Color indefinido,1,0.0001,0.2049,Troncal1.xlsx,Troncal1_10000
4,4,Troncal1_10000.tif,Negro,66,0.0044,13.5245,Troncal1.xlsx,Troncal1_10000
...,...,...,...,...,...,...,...,...
131213,22837,Troncal9_37824.tif,Color indefinido,96,0.0175,100.0015,Troncal9.xlsx,Troncal9_37824
131214,22838,Troncal9_37825.tif,Color indefinido,211,0.0422,99.9985,Troncal9.xlsx,Troncal9_37825
131215,22839,Troncal9_37830.tif,Color indefinido,2,0.0422,100.0000,Troncal9.xlsx,Troncal9_37830
131216,22840,Troncal9_37853.tif,Color indefinido,67,0.0119,84.8092,Troncal9.xlsx,Troncal9_37853


In [117]:
#FILTROS ADICIONALES
print(df_unido.columns)


#excluir
#filtro = ~df_unido['descripcion'].str.contains(r'^(Negro|Azul|Gris|violeta)', case=False, regex=True)
filtro = ~df_unido['descripcion'].str.contains(r'^(Color|Negro|Azul|Gris|violeta|Rojo|Lila|Rosa)', case=False, regex=True)

df_filtrado = df_unido[filtro]

# Agrupar por 'imagen', 'descripcion', 'id' y obtener la suma de 'pixeles' y 'porcentaje_total_imagen'
df_agrupado = df_filtrado.groupby(['imagen', 'descripcion', 'id'], as_index=False).agg({
    'pixeles': 'sum',
    'porcentaje_total_imagen': 'sum'
})

# Agrupar por 'imagen', 'id' y obtener la suma de 'pixeles' y 'porcentaje_total_imagen'
df_agrupado = df_filtrado.groupby(['imagen', 'id'], as_index=False).agg({
    'pixeles': 'sum',
    'porcentaje_total_imagen': 'sum'
})
# Mostrar el DataFrame agrupado
df_agrupado


# Filtrar df_agrupado por 'porcentaje_total_imagen'
df_filtrado_porcentaje = df_agrupado[(df_agrupado['porcentaje_total_imagen'] > 0.3) & (df_agrupado['porcentaje_total_imagen'] < 0.5)]
df_filtrado_porcentaje.rename(columns={'porcentaje_total_imagen': 'anomalies_pct'}, inplace=True)
print(len(df_filtrado_porcentaje.id.value_counts()))

anomalies_excel=r'D:\CORREDORES_V_DATASET\Troncales_anomalies.xlsx'
df_filtrado_porcentaje.to_excel(anomalies_excel)

df_filtrado_porcentaje

Index(['Unnamed: 0', 'imagen', 'descripcion', 'pixeles',
       'porcentaje_total_imagen', 'porcentaje_filtrada', 'archivo_origen',
       'id'],
      dtype='object')


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_18784\1613394666.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtro = ~df_unido['descripcion'].str.contains(r'^(Color|Negro|Azul|Gris|violeta|Rojo|Lila|Rosa)', case=False, regex=True)


336


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_18784\1613394666.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado_porcentaje.rename(columns={'porcentaje_total_imagen': 'anomalies_pct'}, inplace=True)


,imagen,id,pixeles,anomalies_pct
77,Troncal1_10720.tif,Troncal1_10720,5880,0.3462
114,Troncal1_10793.tif,Troncal1_10793,6880,0.4442
117,Troncal1_10807.tif,Troncal1_10807,4372,0.3115
120,Troncal1_10815.tif,Troncal1_10815,7293,0.3018
198,Troncal1_11488.tif,Troncal1_11488,6838,0.3300
...,...,...,...,...
7220,Troncal9_36469.tif,Troncal9_36469,7936,0.4486
7259,Troncal9_36577.tif,Troncal9_36577,5159,0.3105
7417,Troncal9_37604.tif,Troncal9_37604,8919,0.3757
7422,Troncal9_37609.tif,Troncal9_37609,3650,0.4040


In [ ]:
#Chequeo de imagenes por troncal
df_filtrado_porcentaje['troncal'] = df_filtrado_porcentaje['imagen'].astype(str).str.split('_').str[0]
df_filtrado_porcentaje.troncal.value_counts()
df_filtrado_porcentaje[df_filtrado_porcentaje['troncal']=='Troncal2']

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_18784\106100427.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado_porcentaje['troncal'] = df_filtrado_porcentaje['imagen'].astype(str).str.split('_').str[0]


,imagen,id,pixeles,porcentaje_total_imagen,troncal
1603,Troncal2_25287.tif,Troncal2_25287,3746,0.3560,Troncal2
1619,Troncal2_25312.tif,Troncal2_25312,4293,0.3585,Troncal2
1628,Troncal2_25321.tif,Troncal2_25321,6197,0.3330,Troncal2
1629,Troncal2_25322.tif,Troncal2_25322,6981,0.4082,Troncal2
1646,Troncal2_25353.tif,Troncal2_25353,6761,0.3669,Troncal2
...,...,...,...,...,...
2808,Troncal2_29847.tif,Troncal2_29847,5378,0.3752,Troncal2
2891,Troncal2_30176.tif,Troncal2_30176,5184,0.3353,Troncal2
2924,Troncal2_30248.tif,Troncal2_30248,3747,0.4334,Troncal2
3008,Troncal2_31118.tif,Troncal2_31118,10608,0.4564,Troncal2


# Unir shapefiles

In [ ]:
import geopandas as gpd
import pandas as pd
import glob
import os

# Ruta de los archivos .shp
ruta_archivos = r'D:\CORREDORES_V_DATASET\IMAGES\ONLY_ROADS_SAMpavement_terrz19_Z21_BB22\seg_col\Troncal3'

# Buscar todos los archivos *_colores.shp
archivos_shp = glob.glob(os.path.join(ruta_archivos, "*.shp"))

# Cargar todos los archivos en un solo GeoDataFrame
gdfs = []
for archivo in archivos_shp:
    try:
        gdf = gpd.read_file(archivo)
        gdfs.append(gdf)
    except Exception as e:
        print(f"❌ Error cargando {archivo}: {e}")

# Combinar todos
gdf_completo = pd.concat(gdfs, ignore_index=True)

# Asegúrate de que las columnas necesarias existan en tus archivos .shp
columnas_necesarias = ["imagen", "descripcio", "pixeles", "pct_total", "pct_filtra", "geometry"]

if all(col in gdf_completo.columns for col in columnas_necesarias):
    # Agrupar por imagen y descripción
    gdf_agrupado = gdf_completo.groupby(
        ["imagen", "descripcio"], as_index=False
    ).agg({
        "pixeles": "sum",
        "pct_total": "sum",
        "pct_filtra": "sum"
    })

    # Crear un GeoDataFrame usando la geometría original
    gdf_agrupado = gpd.GeoDataFrame(gdf_agrupado, geometry=gdf_completo.geometry)

    # Exportar el GeoDataFrame agrupado por "imagen" y "descripcion" a un archivo .shp
    out_shp_agrupado=ruta_archivos+'.shp'
    gdf_agrupado.to_file(out_shp_agrupado, driver='ESRI Shapefile')

    # Agrupar por imagen
    gdf_agrupado2 = gdf_completo.groupby(
        ["imagen"], as_index=False
    ).agg({
        "pixeles": "sum",
        "pct_total": "sum",
        "pct_filtra": "sum"
    })

    # Crear un GeoDataFrame usando la geometría original
    gdf_agrupado2 = gpd.GeoDataFrame(gdf_agrupado2, geometry=gdf_completo.geometry)

    # Exportar el GeoDataFrame agrupado solo por "imagen" a un archivo .shp
    out_shp_agrupado2 = ruta_archivos+'_colores_agrupados.shp'
    gdf_agrupado2.to_file(out_shp_agrupado2, driver='ESRI Shapefile')

    print("✅ Archivos shapefile exportados correctamente.")
else:
    print("⚠️ Las columnas necesarias no se encontraron en los archivos .shp. Verifica que tus archivos .shp tengan las columnas: ", columnas_necesarias)
